# Tarea 3 — PyTorch

**Nombre del integrante:**  Mario Suso

**Cédula del integrante:** 30.751.403

> Enunciado, rúbrica y nombres de los archivos `.pth` a entregar: ver `README.md` de esta carpeta.
> Antes del commit: *Restart & Run All* debe terminar sin errores y quedar en el menú.

## 1. Imports y constantes

In [1]:
import copy

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets

NOTEBOOK_ID = 2
N_INPUTS = 784
BATCH_SIZE = 128
VAL_SIZE = 6000  # held out from train only when early stopping is on
PATIENCE = 3

CLASS_NAMES = {
    "original": ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                 "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"],
    # output order required by the grader: 0=Top, 1=Bottom, 2=Footwear, 3=Bag
    "simplified": ["Top", "Bottom", "Footwear", "Bag"],
}
# original label (0..9) -> simplified label (0..3)
SIMPLIFIED_MAP = torch.tensor([0, 1, 0, 1, 0, 2, 0, 2, 3, 2])

DEFAULT_OPTIONS = {
    "optimizer": "SGD",
    "lr": 0.01,
    "weight_decay": 0.0,
    "dropout": 0.0,
    "batchnorm": False,
    "early_stopping": False,
}


## 2. Datos (Fashion MNIST original y clases simplificadas)

In [2]:
# Whole dataset fits in memory: pixels scaled to [0, 1] (same as ToTensor), no Normalize
train_set = datasets.FashionMNIST("./data", train=True, download=True)
test_set = datasets.FashionMNIST("./data", train=False, download=True)


def load_data(classification):
    """Return (x_train, y_train, x_test, y_test) for 'original' or 'simplified'."""
    x_train = train_set.data.float() / 255
    x_test = test_set.data.float() / 255
    y_train, y_test = train_set.targets, test_set.targets
    if classification == "simplified":
        y_train, y_test = SIMPLIFIED_MAP[y_train], SIMPLIFIED_MAP[y_test]
    return x_train, y_train, x_test, y_test


## 3. Modelos (rectangular y convolucional)

In [3]:
def hidden_sizes(kind, layers, neurons, n_out):
    """Neurons of each hidden layer: constant (rectangular) or linear funnel (convolutional)."""
    if kind == "rectangular":
        return [neurons] * layers
    return [round(N_INPUTS + (n_out - N_INPUTS) * i / (layers + 1)) for i in range(1, layers + 1)]


def build_model(sizes, n_out, batchnorm, dropout):
    layers = [nn.Flatten()]
    previous = N_INPUTS
    for size in sizes:
        layers.append(nn.Linear(previous, size))
        if batchnorm:
            layers.append(nn.BatchNorm1d(size))
        layers.append(nn.ReLU())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        previous = size
    layers.append(nn.Linear(previous, n_out))
    return nn.Sequential(*layers)


def new_net(classification, kind, layers, neurons, options):
    """A net is a dict: model, hyperparameters (saved in the .pth) and loss history."""
    n_out = len(CLASS_NAMES[classification])
    sizes = hidden_sizes(kind, layers, neurons, n_out)
    hyperparameters = {
        "network_type": kind,
        "hidden_layers": layers,
        "neurons_per_layer": sizes,
        "epochs": 0,
        "classification": classification,
        "optimizations": options,
    }
    model = build_model(sizes, n_out, options["batchnorm"], options["dropout"])
    return {"model": model, "hp": hyperparameters, "losses": []}


## 4. Entrenar, probar, guardar y cargar

In [4]:
OPTIMIZERS = {
    "SGD": lambda params, lr, wd: torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=wd),
    "Adam": lambda params, lr, wd: torch.optim.Adam(params, lr=lr, weight_decay=wd),
    "AdamW": lambda params, lr, wd: torch.optim.AdamW(params, lr=lr, weight_decay=wd),
}


def train(net, epochs):
    model, hp = net["model"], net["hp"]
    opts = hp["optimizations"]
    x, y, _, _ = load_data(hp["classification"])
    if opts["early_stopping"]:
        x_val, y_val = x[-VAL_SIZE:], y[-VAL_SIZE:]
        x, y = x[:-VAL_SIZE], y[:-VAL_SIZE]

    optimizer = OPTIMIZERS[opts["optimizer"]](model.parameters(), opts["lr"], opts["weight_decay"])
    loss_fn = nn.CrossEntropyLoss()
    best_val, best_state, bad_epochs = float("inf"), None, 0

    for _ in range(epochs):
        model.train()
        order = torch.randperm(len(x))
        total = 0.0
        for start in range(0, len(x), BATCH_SIZE):
            idx = order[start:start + BATCH_SIZE]
            optimizer.zero_grad()
            loss = loss_fn(model(x[idx]), y[idx])
            loss.backward()
            optimizer.step()
            total += loss.item() * len(idx)
        net["losses"].append(total / len(x))
        hp["epochs"] += 1
        print(f"epoch {hp['epochs']} - loss {net['losses'][-1]:.4f}")

        if opts["early_stopping"]:
            model.eval()
            with torch.no_grad():
                val_loss = loss_fn(model(x_val), y_val).item()
            if val_loss < best_val:
                best_val, bad_epochs = val_loss, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                bad_epochs += 1
                if bad_epochs >= PATIENCE:
                    print("early stopping")
                    break
    if best_state is not None:
        model.load_state_dict(best_state)

    plt.plot(range(1, len(net["losses"]) + 1), net["losses"], marker="o")
    plt.xlabel("Época")
    plt.ylabel("Pérdida (loss)")
    plt.title("Pérdida por época")
    plt.show()


def test(net):
    model, hp = net["model"], net["hp"]
    _, _, x, y = load_data(hp["classification"])
    names = CLASS_NAMES[hp["classification"]]
    n = len(names)
    model.eval()
    with torch.no_grad():
        pred = model(x).argmax(dim=1)

    # rows = true class, columns = predicted class
    cm = torch.bincount(y * n + pred, minlength=n * n).reshape(n, n).numpy()
    hits = np.diag(cm)
    precision = hits / np.maximum(cm.sum(axis=0), 1)
    recall = hits / np.maximum(cm.sum(axis=1), 1)

    print(f"Accuracy: {hits.sum() / cm.sum():.4f}")
    for i, name in enumerate(names):
        print(f"{name:>12}: precision {precision[i]:.4f}  recall {recall[i]:.4f}")

    plt.imshow(cm, cmap="Blues")
    plt.xticks(range(n), names, rotation=90)
    plt.yticks(range(n), names)
    for i in range(n):
        for j in range(n):
            plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=7)
    plt.xlabel("Predicha")
    plt.ylabel("Real")
    plt.title("Matriz de confusión")
    plt.show()


def default_filename(hp):
    return f"pytorch_fashion_{NOTEBOOK_ID}_{hp['classification']}_{hp['network_type']}.pth"


def save_net(net, path):
    torch.save({"state_dict": net["model"].state_dict(), "hyperparameters": net["hp"]}, path)


def load_net(path):
    checkpoint = torch.load(path, weights_only=True)
    hp = checkpoint["hyperparameters"]
    opts = hp["optimizations"]
    n_out = len(CLASS_NAMES[hp["classification"]])
    model = build_model(hp["neurons_per_layer"], n_out, opts["batchnorm"], opts["dropout"])
    model.load_state_dict(checkpoint["state_dict"])
    return {"model": model, "hp": hp, "losses": []}


## 5. Menú

La última celda debe dejar el menú corriendo (crear/cargar, entrenar, probar, guardar, salir).

In [ ]:
def ask(prompt, default, cast=str):
    raw = input(f"{prompt} [{default}]: ").strip()
    return cast(raw) if raw else default


def ask_yes_no(prompt):
    return input(f"{prompt} (s/N): ").strip().lower() == "s"


def ask_choice(prompt, choices, default):
    value = ask(f"{prompt} ({'/'.join(choices)})", default)
    if value not in choices:
        raise ValueError(f"opción inválida: {value}")
    return value


def create_from_menu():
    classification = ask_choice("Clasificación", ["original", "simplified"], "original")
    kind = ask_choice("Tipo de red", ["rectangular", "convolutional"], "rectangular")
    layers = ask("Número de capas ocultas", 2, int)
    neurons = ask("Neuronas por capa", 512, int) if kind == "rectangular" else None
    options = dict(DEFAULT_OPTIONS)
    if ask_yes_no("¿Configurar optimizaciones?"):
        options["optimizer"] = ask_choice("Optimizador", list(OPTIMIZERS), "SGD")
        options["lr"] = ask("Learning rate", 0.01, float)
        options["weight_decay"] = ask("Weight decay", 0.0, float)
        options["dropout"] = ask("Dropout (0 = desactivado)", 0.0, float)
        options["batchnorm"] = ask_yes_no("¿Batch normalization?")
        options["early_stopping"] = ask_yes_no("¿Early stopping?")
    net = new_net(classification, kind, layers, neurons, options)
    print(f"Red creada: 784 -> {net['hp']['neurons_per_layer']} -> {len(CLASS_NAMES[classification])}")
    return net


MENU = """
===== Menú =====
1. Crear nueva red
2. Cargar red desde archivo
3. Entrenar
4. Probar
5. Guardar
6. Salir"""

net = None
while True:
    print(MENU)
    choice = input("Opción: ").strip()
    try:
        if choice == "1":
            net = create_from_menu()
        elif choice == "2":
            net = load_net(ask("Archivo .pth", f"pytorch_fashion_{NOTEBOOK_ID}_original_rectangular.pth"))
            print(f"Red cargada ({net['hp']['epochs']} épocas acumuladas).")
        elif choice == "6":
            print("Hasta luego.")
            break
        elif choice in ("3", "4", "5"):
            if net is None:
                print("Primero crea o carga una red.")
            elif choice == "3":
                train(net, ask("Número de épocas", 10, int))
            elif choice == "4":
                test(net)
            else:
                path = ask("Archivo .pth", default_filename(net["hp"]))
                save_net(net, path)
                print(f"Guardada en {path}")
        else:
            print("Opción inválida.")
    except (ValueError, OSError, KeyError, RuntimeError) as error:
        print(f"Error: {error}")



===== Menú =====
1. Crear nueva red
2. Cargar red desde archivo
3. Entrenar
4. Probar
5. Guardar
6. Salir
